In [2]:
!pip install ultralytics
!pip install roboflow



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\User\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


# get dataset

In [3]:
%pip install roboflow
import roboflow
print(roboflow.__version__)

Note: you may need to restart the kernel to use updated packages.
1.4.2



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from dotenv import load_dotenv
import os
from roboflow import Roboflow

load_dotenv()

api_key = os.getenv("api_key_robflow")

rf = Roboflow(api_key=api_key)
project = rf.workspace("roboflow-jvuqo").project("football-players-detection-3zvbc")
version = project.version(1)
dataset = version.download("yolov5")

loading Roboflow workspace...
loading Roboflow project...


In [5]:
dataset.location

'c:\\Users\\User\\Documents\\match_analyser_yahya\\training\\football-players-detection-1'

# fix dataset paths


In [6]:
import os, yaml

root = dataset.location

def find_split_dir(root, split_names):
    """Walk the dataset folder and find the 'images' directory belonging to a given split,
    regardless of how deeply Roboflow (or a manual move) nested it, or whether it's named 'valid' or 'val'."""
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == "images":
            parent = os.path.basename(os.path.dirname(dirpath))
            if parent in split_names:
                return dirpath
    return None

train_dir = find_split_dir(root, ["train"])
valid_dir = find_split_dir(root, ["valid", "val", "validation"])
test_dir  = find_split_dir(root, ["test"])

print("train:", train_dir)
print("valid:", valid_dir)
print("test: ", test_dir)

assert train_dir and valid_dir, "Could not locate train/valid image folders — check the folder tree with os.walk(root)."

data_yaml_path = os.path.join(root, "data.yaml")

with open(data_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

# Use absolute paths directly and drop 'path', so nothing gets concatenated/doubled
data_cfg.pop("path", None)
data_cfg["train"] = train_dir
data_cfg["val"] = valid_dir
if test_dir:
    data_cfg["test"] = test_dir

with open(data_yaml_path, "w") as f:
    yaml.dump(data_cfg, f)

print("\nFixed data.yaml:")
print(yaml.dump(data_cfg))


train: c:\Users\User\Documents\match_analyser_yahya\training\football-players-detection-1\football-players-detection-1\train\images
valid: c:\Users\User\Documents\match_analyser_yahya\training\football-players-detection-1\football-players-detection-1\valid\images
test:  c:\Users\User\Documents\match_analyser_yahya\training\football-players-detection-1\football-players-detection-1\test\images

Fixed data.yaml:
names:
- ball
- goalkeeper
- player
- referee
nc: 4
roboflow:
  license: CC BY 4.0
  project: football-players-detection-3zvbc
  url: https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc/dataset/1
  version: 1
  workspace: roboflow-jvuqo
test: c:\Users\User\Documents\match_analyser_yahya\training\football-players-detection-1\football-players-detection-1\test\images
train: c:\Users\User\Documents\match_analyser_yahya\training\football-players-detection-1\football-players-detection-1\train\images
val: c:\Users\User\Documents\match_analyser_yahya\training\foo

# training

In [7]:
!yolo task=detect mode=train model=yolov5x.pt data={dataset.location}/data.yaml epochs=100 imgsz=640

^C
